In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sallerroig/court-seg")

print("Path to dataset files:", path)

c:\Users\524ha\anaconda3\envs\seg\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 38.1M/38.1M [00:02<00:00, 15.9MB/s]

Extracting files...


Path to dataset files: C:\Users\524ha\.cache\kagglehub\datasets\sallerroig\court-seg\versions\1


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gabrielvanzandycke/deepsport-dataset")

print("Path to dataset files:", path)


c:\Users\524ha\anaconda3\envs\seg\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 2.52G/2.52G [01:58<00:00, 22.8MB/s]

Extracting files...


Path to dataset files: C:\Users\524ha\.cache\kagglehub\datasets\gabrielvanzandycke\deepsport-dataset\versions\8


In [1]:
import torch
print("Torch version:", torch.__version__)

Torch version: 2.8.0+cu128


In [3]:
import cv2
import numpy as np
import os
import random
from glob import glob

# --- AYARLAR ---
# Lütfen bu yolları, bir önceki script'teki 
# OUTPUT_DIR yolunuza göre güncelleyin.
# (Bir önceki script'inize göre ayarladım)
BASE_OUTPUT_DIR = '../data/seg/basketball_players_and_ball'

IMAGE_DIR = os.path.join(BASE_OUTPUT_DIR, 'images')
LABEL_DIR = os.path.join(BASE_OUTPUT_DIR, 'labels')

# Doğrulanmış maskelerin kaydedileceği yer
OUTPUT_VERIFICATION_DIR = os.path.join(BASE_OUTPUT_DIR, 'verification_output')

# Kaç adet rastgele örnek görmek istiyorsunuz?
NUM_SAMPLES = 5
# --- AYARLAR SONU ---

def draw_yolo_masks(img_path, label_path, output_path):
    """
    Bir görüntüyü ve YOLO .txt etiketini alır,
    maskeleri görüntü üzerine çizer ve kaydeder.
    """
    
    # Görüntüyü oku
    image = cv2.imread(img_path)
    if image is None:
        print(f"Hata: Görüntü okunamadı {img_path}")
        return
        
    h, w = image.shape[:2]
    
    # Yarı saydam maske için bir kopya oluştur
    overlay = image.copy()
    
    try:
        # Etiket dosyasını oku
        with open(label_path, 'r') as f:
            lines = f.readlines()
            
        if not lines:
            tqdm.write(f"Not: {label_path} boş (resimde nesne yok).")

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 3: # En az class_id ve bir x,y çifti olmalı
                continue
            
            # class_id = int(parts[0]) # Şu an kullanmıyoruz ama burada
            
            # Poligon noktalarını al (string'den float'a)
            polygon_normalized = np.array(parts[1:], dtype=np.float32)
            
            # (x1, y1, x2, y2...) -> [[x1, y1], [x2, y2], ...]
            polygon_normalized = polygon_normalized.reshape(-1, 2)
            
            # Koordinatları de-normalize et (0-1 aralığından piksel aralığına)
            polygon_pixels = polygon_normalized.copy()
            polygon_pixels[:, 0] *= w  # X koordinatları
            polygon_pixels[:, 1] *= h  # Y koordinatları
            
            # CV2'nin istediği formata getir (int32)
            polygon_pixels = polygon_pixels.astype(np.int32)
            
            # Poligonu overlay üzerine kırmızı renkte (0,0,255) doldur
            cv2.fillPoly(overlay, [polygon_pixels], (0, 0, 255)) # BGR formatında Kırmızı

    except FileNotFoundError:
        print(f"Uyarı: Etiket dosyası bulunamadı {label_path}")
        # Etiket dosyası olmasa bile resmi kaydet (negatif örnek)
    except Exception as e:
        print(f"Maske çizilirken hata oluştu {label_path}: {e}")
        return

    # Orijinal resim ile maskeli overlay'i birleştir (%50 saydamlık)
    alpha = 0.5
    cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0, image)
    
    # Çizilmiş görüntüyü kaydet
    cv2.imwrite(output_path, image)


# --- Ana Script ---
if __name__ == "__main__":
    
    # 1. Çıktı klasörünü oluştur
    os.makedirs(OUTPUT_VERIFICATION_DIR, exist_ok=True)
    print(f"Doğrulama görüntüleri şuraya kaydedilecek: {OUTPUT_VERIFICATION_DIR}")

    # 2. Tüm etiket dosyalarının listesini al
    all_label_files = glob(os.path.join(LABEL_DIR, '*.txt'))
    
    if not all_label_files:
        print(f"Hata: '{LABEL_DIR}' içinde hiç .txt dosyası bulunamadı.")
    else:
        # 3. Rastgele örnekler seç
        # Eğer toplam dosya sayısı NUM_SAMPLES'dan azsa, hepsi seçilir
        num_to_sample = min(NUM_SAMPLES, len(all_label_files))
        if num_to_sample == 0:
             print("İşlenecek örnek bulunamadı.")
        else:
            selected_files = random.sample(all_label_files, num_to_sample)
            
            print(f"Toplam {len(all_label_files)} etiketten {num_to_sample} rastgele örnek işleniyor...")
            
            for label_path in selected_files:
                base_name = os.path.basename(label_path)
                img_name = base_name.replace('.txt', '.png')
                
                img_path = os.path.join(IMAGE_DIR, img_name)
                output_path = os.path.join(OUTPUT_VERIFICATION_DIR, img_name)
                
                if not os.path.exists(img_path):
                    print(f"Hata: {img_path} bulunamadı, {label_path} atlanıyor.")
                    continue
                    
                draw_yolo_masks(img_path, label_path, output_path)

            print(f"\nİşlem tamamlandı!")
            print(f"Lütfen sonuçları kontrol etmek için '{OUTPUT_VERIFICATION_DIR}' klasörünü açın.")

Doğrulama görüntüleri şuraya kaydedilecek: ../data/seg/basketball_players_and_ball\verification_output
Toplam 310 etiketten 5 rastgele örnek işleniyor...

İşlem tamamlandı!
Lütfen sonuçları kontrol etmek için '../data/seg/basketball_players_and_ball\verification_output' klasörünü açın.


In [1]:
import numpy as np
data = np.load("filename.npy", allow_pickle = True).item()


FileNotFoundError: [Errno 2] No such file or directory: 'filename.npy'

In [1]:
from super_gradients.training import models
from super_gradients.common.object_names import Models
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = models.get(Models.YOLO_NAS_POSE_L, pretrained_weights="coco_pose")  # büyük model örneği
model.to(device)
model.eval()

# Ardından bir resim veya video için infer yapılsın:
result = model.predict('sahne.jpg', conf=0.25)  # örnek
result.save('output.jpg')


ModuleNotFoundError: No module named 'super_gradients'